# Experiment design, controls and ablations: deterministic example

Match the return indexes and provenance for two synthetic arms. Declare the policy as the single intervention.

In [1]:
from math import isclose

import pandas as pd

from evaluation.metrics.matched_comparison import compare_matched_returns

timestamps = pd.date_range("2000-01-01", periods=4, freq="D", tz="UTC")
treatment = pd.Series([0.03, 0.00, 0.04, 0.01], index=timestamps)
control = pd.Series([0.02, -0.01, 0.03, 0.00], index=timestamps)
shared = {
    "dataset_sha256": "fixture-data",
    "split_id": "fixture-split",
    "features_sha256": "fixture-features",
    "reward_sha256": "fixture-reward",
    "costs_sha256": "fixture-costs",
    "execution_sha256": "fixture-execution",
    "timesteps": 4,
    "seed": 42,
    "initial_capital": 100,
    "model_settings_sha256": "fixture-model",
    "environment_sha256": "fixture-environment",
    "evaluation_start": timestamps[0].isoformat(),
    "evaluation_end": timestamps[-1].isoformat(),
}
treatment_record = {**shared, "policy_sha256": "fixture-treatment"}
control_record = {**shared, "policy_sha256": "fixture-control"}
interval = compare_matched_returns(
    treatment,
    control,
    treatment_record,
    control_record,
    intervention_field="policy_sha256",
    n_boot=32,
)
assert isclose(interval.estimate, 0.01)
print(f"Synthetic matched difference: {interval.estimate:.2%}")

try:
    compare_matched_returns(
        treatment,
        control,
        treatment_record,
        {**control_record, "costs_sha256": "changed"},
        intervention_field="policy_sha256",
        n_boot=32,
    )
except ValueError as error:
    assert "costs_sha256" in str(error)
    print("Second intervention rejected:", error)
else:
    raise AssertionError("The changed cost identity must fail matching.")

Synthetic matched difference: 1.00%
Second intervention rejected: Missing or mismatched comparison provenance: costs_sha256


The identity strings are fixture labels. Real studies need hashes from executed runs and an independent record of each arm. The helper checks equality and alignment for this path.